### AmazingData

In [1]:
# sudo sh -c 'ifconfig wlp0s20f3 down && ifconfig wlp0s20f3 hw ether 30:E3:A4:6E:24:CC && ifconfig wlp0s20f3 up'

In [2]:
import AmazingData as ad
import pandas as pd
import os, gc, datetime, shutil
from pathlib import Path
import duckdb
from dotenv import load_dotenv
import polars as pl



load_dotenv('conf/.env')
tgw_uname = os.environ.get('TGW_USER')
tgw_upass = os.environ.get('TGW_PASS')
tgw_server = os.environ.get('TGW_HOST')
tgw_datadir = './data/amazing/'
# syd_datadir = 'D:/Data/SynologyDrive/'
# tgw_datadir = 'D:/Script/tgwStock/data/'
tgw_islocal = False
tgw_ziptype = 'zstd'


def release_memory(var_list: list):
    for v in var_list: del v
    gc.collect()


def log_status(process, status):
    print(f'[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {process} - status: {status}')

In [3]:
try:
    ad.logout(username=tgw_uname)
finally:
    ad.login(username= tgw_uname, password=tgw_upass,host=tgw_server,port=8600)

    bdo = ad.BaseData()
    calendar = bdo.get_calendar()
    today = calendar[-1]
    local_today = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(hours=8)
    market_close = str(today) < local_today.strftime('%Y%m%d')
    print(f'{today} market close status: {market_close}')

    mdo=ad.MarketData(calendar) 
    ido = ad.InfoData()

TGW Logon information:  : 
logon json :  {"Id":0,"SubscribeLimitNum":0,"PushBandwidth":2048,"QueryBandwidth":2048,"TotalWeekFlow":1000000000,"UsedWeekFlow":0.31,"Token":"53746f00-00d3-4e79-a56f-fae6c16a60c6","PermissionCode":"2|3|4|6|7|11|12|13|29|30|31|32|33|16|17|18|19|20|21|22|23|24|25|26|27|28","FunctionIdPermission":["A010010001","A010010002","A010010003","A010010004","A010010005","A010010006","A010010007","A010010007_NEW","A010010008","A010010008_NEW","A010010009","A010010009_NEW","A010010010","A010010011","A010020001","A010020002","A010030001","A010030002","A010030003","A010030004","A010040001","A010040002","A010040003","A010040004","A010040005","A010050001","A010050001_CDR","A010050002","A010050002_CDR","A010050003","A010050003_CDR","A010050004","A010050005","A010050006","A010050007","A010050008","A010050009","A010060001","A010060002","A010060003","A010060004","A010060005","A010060006","A010060007","A010061001","A010061002","A010061003","A010070001","A010070002","A010070002_inn

In [4]:
def download_base(data_type:str, **kwargs):
    func_dict = {
        'info_index_detail': ido.get_index_constituent,
        'info_index_weight': ido.get_index_weight,
        'info_industry_basic': ido.get_industry_base_info,  
        'info_industry_detail': ido.get_industry_constituent,
        'info_industry_weight': ido.get_industry_weight,
        'finance_balance_sheet': ido.get_balance_sheet,
        'finance_cash_flow': ido.get_cash_flow,
        'finance_income': ido.get_income,        
        'rank_longhubang': ido.get_long_hu_bang,
        'margin_summary': ido.get_margin_summary,
        'margin_detail': ido.get_margin_detail,
        'share_holdertop': ido.get_share_holder,
        'share_holdernum': ido.get_holder_num,             
        'equity_restricted': ido.get_equity_restricted,
        'equity_structure': ido.get_equity_structure,
        'equity_dividend': ido.get_dividend,
        'profit_express': ido.get_profit_express,
        'profit_notice': ido.get_profit_notice,
        'stock_status': ido.get_history_stock_status,
        'option_basic_info': ido.get_option_basic_info,
        'option_std_ctr_specs': ido.get_option_std_ctr_specs,
        'option_mon_ctr_specs': ido.get_option_mon_ctr_specs,
        'treasury_yield':ido.get_treasury_yield,
    }
    if data_type not in func_dict.keys(): 
        raise ValueError(f'{data_type} is not in {func_dict.keys()}')

    # fname = f'{kwargs.get('local_path','./data/')}/{data_type}_{today}.parquet'
    fname = f'{kwargs.get('local_path','./data/')}/{data_type}_history.parquet'
    df = func_dict[data_type](**kwargs)

    if type(df)==type(dict()):
        pd.concat([v for v in df.values() if v is not None]).to_parquet(fname, compression=tgw_ziptype)
    elif type(df)==type(pd.DataFrame()):
        df.to_parquet(fname, compression=tgw_ziptype)
    else:
        raise TypeError(f'saving [{type(df)}] to parquet is not defined')

    print(pd.read_parquet(fname).shape, fname)

    release_memory([df])

### STOCK BASIC

In [5]:
fname = f'{tgw_datadir}/info_code_basic.parquet'
bdo.get_code_info(security_type='EXTRA_STOCK_A').reset_index().to_parquet(fname, compression=tgw_ziptype)
print(fname, datetime.datetime.now())

./data/amazing//info_code_basic.parquet 2026-04-21 15:13:22.446408


In [6]:
code_list = bdo.get_code_list(security_type='EXTRA_STOCK_A')

In [7]:
fname = f'{tgw_datadir}/info_stock_basic.parquet'
ido.get_stock_basic(code_list).reset_index().to_parquet(fname, compression=tgw_ziptype)
print(fname, datetime.datetime.now())

证券信息数据 下载完成
./data/amazing//info_stock_basic.parquet 2026-04-21 15:17:24.735685


In [8]:
df_factor = bdo.get_backward_factor(code_list, local_path=tgw_datadir, is_local=False)
df_factor = df_factor.unstack().reset_index()
df_factor.columns=['instrument','datetime','backward_factor']
df_factor.to_parquet(f'{tgw_datadir}/info_stock_factor.parquet', compression=tgw_ziptype)
print(df_factor['datetime'].unique().shape, df_factor['datetime'].min(), '\t', df_factor['datetime'].max())

print('Date ERROR') if df_factor['datetime'].max().strftime('%Y%m%d') != str(today) else print("Completed")

(8625,) 1990-12-19 00:00:00 	 2026-04-21 00:00:00
Completed


## OHLCVF

In [9]:
config = {
    'd': {'begin_date': today, 'end_date': today},
    'p': {'begin_date': calendar[-2], 'end_date': calendar[-2]},
    'w': {'begin_date': calendar[-5], 'end_date': calendar[-1]},
    'a': {'begin_date': calendar[0], 'end_date': calendar[-1]},
}

dtargs = config['d']
# dtargs = {'begin_date': 20251201, 'end_date': 20251231}
print(dtargs)

{'begin_date': 20260421, 'end_date': 20260421}


In [10]:
def download_price(sec_type, begin_date, end_date, period:int=0):
    if period==0: period = ad.constant.Period.day.value

    security_type = sec_type.upper()
    code_list = bdo.get_code_list(security_type)

    df_kline = mdo.query_kline (code_list, begin_date=begin_date, end_date=end_date, period=period)
    df_ohlcv = pd.concat([v for v in df_kline.values() if v is not None])

    if 'STOCK' in sec_type:
        df_factor = pd.read_parquet(f'{tgw_datadir}/info_stock_factor.parquet')
        bidx = f'{str(begin_date)[:4]}-{str(begin_date)[4:6]}-{str(begin_date)[-2:]}'
        eidx = f'{str(end_date)[:4]}-{str(end_date)[4:6]}-{str(end_date)[-2:]}'
        df_factor= df_factor[df_factor['datetime'].between(bidx, eidx)]
        df_factor.columns=['code','kline_time', 'backward_factor']
        df_ohlcv = pd.merge(df_ohlcv, df_factor, on=['kline_time','code'], how='left')
        release_memory([df_factor])
        
    file_type= sec_type.split('_')[1].lower()
    if period == ad.constant.Period.day.value:
        file_name = f'extra_{file_type}_{end_date}.parquet'
    else:
        file_name = f'extra_{file_type}_{period}_{end_date}.parquet'

    df_ohlcv.to_parquet(Path(tgw_datadir).joinpath(file_name),compression=tgw_ziptype, index=False)
    print(df_ohlcv.shape, file_name, df_ohlcv['kline_time'].max())
    release_memory([df_kline, df_ohlcv]) 
    # print(f"Copying {file_name} to Synology Drive...")
    # shutil.copy2(Path(tgw_datadir).joinpath(file_name), Path(syd_datadir))

In [11]:
print(datetime.datetime.now())

data_list = ['EXTRA_STOCK_A', 'EXTRA_INDEX_A','EXTRA_ETF']
for data in data_list: download_price(data, **dtargs)

2026-04-21 15:20:38.659080


(5509, 9) extra_stock_20260421.parquet 2026-04-21 00:00:00
(566, 8) extra_index_20260421.parquet 2026-04-21 00:00:00
(1481, 8) extra_etf_20260421.parquet 2026-04-21 00:00:00


### INFOBASE

In [12]:
data_list = []
# data_list.extend(['finance_cash_flow','finance_balance_sheet', 'finance_income'])
# data_list.extend(['share_holdertop', 'share_holdernum'])
# data_list.extend(['equity_structure', 'equity_dividend','equity_restricted'])
# data_list.extend(['margin_summary', 'margin_detail'])
# data_list.extend(['rank_longhubang'])
# data_list.extend(['profit_express','profit_notice'])
# data_list.extend(['stock_status'])
data_list.extend(['equity_dividend','margin_summary', 'margin_detail'])


code_list = bdo.get_code_list(security_type='EXTRA_STOCK_A')
for data in data_list: 
    log_status(process=data, status='start')
    try: 
        # kwargs = {'code_list': code_list, 'begin_date':calendar[-4], 'end_date':today}
        kwargs = {'code_list': code_list, 'local_path':tgw_datadir, 'is_local':False}        
        if data == 'margin_summary': kwargs.pop('code_list')
        download_base(data_type=data, **kwargs)
    except Exception as e: print("ERROR:", e)
    log_status(process=data, status='completed')


[2026-04-21 15:22:45] equity_dividend - status: start
分红数据 下载完成
(138928, 24) ./data/amazing//equity_dividend_history.parquet
[2026-04-21 15:23:27] equity_dividend - status: completed
[2026-04-21 15:23:27] margin_summary - status: start
融资融券成交汇总数据 下载完成
(8568, 8) ./data/amazing//margin_summary_history.parquet
[2026-04-21 15:23:32] margin_summary - status: completed
[2026-04-21 15:23:32] margin_detail - status: start
./data/amazing/infodata/margin_detail/  本地无600361.SH “融资融券交易明细数据”
./data/amazing/infodata/margin_detail/  本地无603535.SH “融资融券交易明细数据”
./data/amazing/infodata/margin_detail/  本地无600576.SH “融资融券交易明细数据”
./data/amazing/infodata/margin_detail/  本地无605365.SH “融资融券交易明细数据”
./data/amazing/infodata/margin_detail/  本地无603333.SH “融资融券交易明细数据”
./data/amazing/infodata/margin_detail/  本地无603679.SH “融资融券交易明细数据”
./data/amazing/infodata/margin_detail/  本地无603073.SH “融资融券交易明细数据”
./data/amazing/infodata/margin_detail/  本地无601921.SH “融资融券交易明细数据”
./data/amazing/infodata/margin_detail/  本地无603315.SH “

### Treasury Yield

In [13]:
data_type = 'treasury_yield'
data_list = []
for t in ['m3', 'm6', 'y1', 'y2', 'y3', 'y5', 'y7', 'y10', 'y30']:
    data_list.extend([f'{data_type}_{t}'])

for data in data_list: 
    code_list = data.split('_')[-1]
    log_status(process=data, status='start')
    try: 
        kwargs = {'code_list': [code_list], 'local_path':tgw_datadir, 'is_local':True}
        download_base(data_type=data_type, **kwargs)
        df = pd.read_parquet(f'./data/amazing//{data_type}_history.parquet')
        df['TERM']=code_list
        df.to_parquet(f'./data/amazing//{data_type}_history_{code_list}.parquet', index=False)
        os.remove(f'./data/amazing//{data_type}_history.parquet')
    except Exception as e: print("ERROR:", e)
    log_status(process=data, status='completed')

pl.read_parquet(f'./data/amazing//{data_type}_history_*.parquet') \
    .write_parquet(f'./data/amazing//{data_type}_history.parquet')

for f in Path('.').glob(f'./data/amazing//{data_type}_history_*.parquet'):
    os.remove(f)

[2026-04-15 15:25:52] treasury_yield_m3 - status: start
(1631, 2) ./data/amazing//treasury_yield_history.parquet
[2026-04-15 15:25:53] treasury_yield_m3 - status: completed
[2026-04-15 15:25:53] treasury_yield_m6 - status: start
(5400, 2) ./data/amazing//treasury_yield_history.parquet
[2026-04-15 15:25:53] treasury_yield_m6 - status: completed
[2026-04-15 15:25:53] treasury_yield_y1 - status: start
(9448, 2) ./data/amazing//treasury_yield_history.parquet
[2026-04-15 15:25:53] treasury_yield_y1 - status: completed
[2026-04-15 15:25:53] treasury_yield_y2 - status: start
(13485, 2) ./data/amazing//treasury_yield_history.parquet
[2026-04-15 15:25:53] treasury_yield_y2 - status: completed
[2026-04-15 15:25:53] treasury_yield_y3 - status: start
(17510, 2) ./data/amazing//treasury_yield_history.parquet
[2026-04-15 15:25:53] treasury_yield_y3 - status: completed
[2026-04-15 15:25:53] treasury_yield_y5 - status: start
(21558, 2) ./data/amazing//treasury_yield_history.parquet
[2026-04-15 15:25:5

### WEIGHT

In [14]:
data_list = []
data_list.extend(['info_index_detail','info_index_weight'])


code_list = bdo.get_code_list(security_type='EXTRA_INDEX_A')
for data in data_list: 
    log_status(process=data, status='start')    
    try: 
        if data=='info_index_weight': code_list = ['000016.SH','000300.SH','000905.SH','000906.SH','000852.SH']
        kwargs = {'code_list': code_list, 'local_path':tgw_datadir, 'is_local':False}        
        # if data == 'margin_summary': kwargs.pop('code_list')
        download_base(data_type=data, **kwargs)
    except Exception as e: print("ERROR:", e)
    log_status(process=data, status='completed')

[2026-04-15 15:26:08] info_index_detail - status: start
指数成分股数据 下载完成
(294350, 5) ./data/amazing//info_index_detail_history.parquet
[2026-04-15 15:26:25] info_index_detail - status: completed
[2026-04-15 15:26:25] info_index_weight - status: start
指数权重数据 下载完成
(9077194, 9) ./data/amazing//info_index_weight_history.parquet
[2026-04-15 15:26:53] info_index_weight - status: completed


In [15]:
data_list = []
data_list.extend(['info_industry_basic'])

code_list = ['000016.SH','000300.SH','000905.SH','000906.SH','000852.SH']
for data in data_list: 
    log_status(process=data, status='start')
    # if data=='info_index_weight': code_list = ['000016.SH','000300.SH','000905.SH','000906.SH','000852.SH']
    try: 
        kwargs = {'code_list': code_list, 'local_path':tgw_datadir, 'is_local':False}        
        if data == 'info_industry_basic': kwargs.pop('code_list')
        download_base(data_type=data, **kwargs)
    except Exception as e: print("ERROR:", e)
    log_status(process=data, status='completed')


[2026-04-15 15:26:53] info_industry_basic - status: start
(511, 8) ./data/amazing//info_industry_basic_history.parquet
[2026-04-15 15:26:54] info_industry_basic - status: completed


In [16]:
data_list = []
data_list.extend(['info_industry_detail', 'info_industry_weight'])
# data_list.extend(['info_industry_basic'])

code_list = list(pd.read_parquet('./data/amazing/info_industry_basic_history.parquet')['INDEX_CODE'].unique())
for data in data_list: 
    log_status(process=data, status='start')
    # if data=='info_index_weight': code_list = ['000016.SH','000300.SH','000905.SH','000906.SH','000852.SH']
    try: 
        kwargs = {'code_list': code_list, 'local_path':tgw_datadir, 'is_local':False}        
        # if data == 'info_industry_basic': kwargs.pop('code_list')
        download_base(data_type=data, **kwargs)
    except Exception as e: print("ERROR:", e)
    log_status(process=data, status='completed')

[2026-04-15 15:26:54] info_industry_detail - status: start
行业成分股数据 下载完成
(23231, 5) ./data/amazing//info_industry_detail_history.parquet
[2026-04-15 15:26:58] info_industry_detail - status: completed
[2026-04-15 15:26:58] info_industry_weight - status: start
./data/amazing/infodata/industry_weight/  本地无851832.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无851833.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无852061.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无801207.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无852071.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无801216.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无852161.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无801217.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无852171.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无852181.SI “行业权重数据”
./data/amazing/infodata/industry_weight/  本地无852184.SI “行业权重数据”
./data/amazing/infodata/industry_weigh

## File Merge

In [3]:
import polars as pl
import os
import glob



data_list = ['extra_stock', 'extra_index','extra_etf']
# data_list = ['extra_stock']
for data in data_list: 
    print(data)
    pl.read_parquet(f'/media/datax/data/SynologyDrive/{data}_*.parquet') \
        .write_parquet(f'/media/datax/data/SynologyDrive/{data}_all.parquet')

    dl_all = pl.read_parquet(f'/media/datax/data/SynologyDrive/{data}_all.parquet')['kline_time'].unique()
    print('all:', len(dl_all), dl_all.min(), dl_all.max())

    dl_his = pl.read_parquet(f'/media/datax/data/SynologyDrive/{data}_history.parquet')['kline_time'].unique()
    print('his:', len(dl_his), dl_his.min(), dl_his.max())
    print('diff:', len(dl_all) - len(dl_his))

    if (dl_all.len() > dl_his.len()) & (dl_all.max() > dl_his.max()):
        os.rename(
            f'/media/datax/data/SynologyDrive/{data}_all.parquet', 
            f'/media/datax/data/SynologyDrive/{data}_history.parquet'
        )
        for file in glob.glob(f'/media/datax/data/SynologyDrive/{data}_2*.parquet'):
            os.remove(file)

extra_stock
all: 3218 2013-01-04 00:00:00 2026-04-08 00:00:00
his: 3213 2013-01-04 00:00:00 2026-03-31 00:00:00
diff: 5
extra_index
all: 3219 2013-01-04 00:00:00 2026-04-08 00:00:00
his: 3214 2013-01-04 00:00:00 2026-03-31 00:00:00
diff: 5
extra_etf
all: 3218 2013-01-04 00:00:00 2026-04-08 00:00:00
his: 3213 2013-01-04 00:00:00 2026-03-31 00:00:00
diff: 5
